# RAG Query Pipeline (`query_nbk`)

End-to-end **query-time** RAG pipeline over the `optimized_*` data family.

```
query
  -> embed (DMRetriever-33M)
  -> vector search  +  BM25 keyword search
  -> RRF fusion                 (dbx_hybrid_search)
  -> cross-encoder rerank       (dbx_rerank)
  -> top_k context
  -> LLM generation             (LLMs_call)
  -> answer  [-> optional batch evaluation]
```

Reusable logic lives in `scripts/`. One-time data/index builds live in `setup_nbk`.


In [ ]:
# --- Make scripts/ importable ---
# In Databricks Repos the repo root is on sys.path; locate the scripts/ subdir there
# (works the same when running from the repo root locally).
import os, sys

_cands = [os.path.join(p, "scripts") for p in (list(sys.path) + [os.getcwd()])]
for _cand in _cands:
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import warnings
warnings.filterwarnings("ignore")

import markdown
from pyspark.sql import functions as F

from dbx_hybrid_search import hybrid_search_rrf
from dbx_rerank import rerank_dataframe
from LLMs_call import build_messages, dbx_llm_chat, ask_gpt, load_api_key
from rag_pipeline import rag_pipe_main


def display_MD(md_text):
    # Render Markdown answer nicely in a Databricks notebook
    displayHTML(markdown.markdown(md_text, extensions=["fenced_code", "tables"]))

## Config

All retrieval targets use the `optimized_*` family only.

In [ ]:
query = "How did Harris County estimate the number of homes that flooded after Hurricane Harvey, and what data sources did they use?"

# Retrieval / fusion / rerank parameters
TOP_EACH     = 100   # candidates pulled from each retriever before fusion
TOP_N        = 20    # fused candidates kept after RRF (fed into rerank)
RERANK_TOP_K = 10    # final chunks kept after cross-encoder rerank (context for LLM)
RRF_K        = 20    # RRF constant

# Available Databricks-served LLMs
dbx_llms = [
    "databricks-claude-opus-4-6",
    "databricks-claude-sonnet-4-5",
    "databricks-claude-haiku-4-5",
    "databricks-claude-opus-4-5",
    "databricks-claude-opus-4-1",
    "databricks-claude-sonnet-4",
    "databricks-gpt-oss-120b",
    "databricks-gpt-oss-20b",
    "databricks-llama-4-maverick",
    "databricks-gemma-3-12b",
    "databricks-meta-llama-3-1-8b-instruct",
    "databricks-meta-llama-3-3-70b-instruct",
]

## Step 1 — Hybrid retrieval (Vector + BM25 -> RRF fusion)

In [ ]:
fused_df = hybrid_search_rrf(
    spark,
    query,
    top_each=TOP_EACH,
    top_n=TOP_N,
    rrf_k=RRF_K,
    return_with_text=True,
)
display(fused_df)

## Step 2 — Cross-encoder rerank

Re-scores the fused `(query, chunk)` pairs and keeps the most relevant `RERANK_TOP_K`.
Requires the reranker model in the UC Volume (see `setup_nbk` step 0).

In [ ]:
reranked_df = rerank_dataframe(spark, fused_df, query, top_k=RERANK_TOP_K)
display(reranked_df)

## Step 3 — LLM generation

`rag_pipe_main` runs the whole pipeline (retrieve -> fuse -> rerank -> generate) in one call.
Switch backends with `use_dbx_model`.

In [ ]:
# Databricks-served model
reranked_df, answer = rag_pipe_main(
    spark, query,
    llm_model="databricks-llama-4-maverick",
    use_dbx_model=True,
    temperature=0.2,
    top_each=TOP_EACH, top_n=TOP_N, rerank_top_k=RERANK_TOP_K, rrf_k=RRF_K,
)
display_MD(answer)

In [ ]:
# OpenAI model (reads gpt_api_key from config.yaml)
# reranked_df, answer = rag_pipe_main(
#     spark, query,
#     llm_model="gpt-4o-mini",
#     use_dbx_model=False,
#     temperature=0.2,
# )
# display_MD(answer)

## (Optional) Batch evaluation

Runs the pipeline over a fixed QA set across several models and writes the answers
to `tdis_dev_data_catalog.tdir.qa_dict_eval` for review.

In [ ]:
qa_dict = [
    {"question": "How did the 2021 winter storm impact Austin's infrastructure in terms of power and electricity",
     "fact": "Approximately 40% of Austin Energy customers were without power.",
     "fact_source": "2021 Winter Storm Uri After-Action Review"},
    {"question": "What were the major barriers to effective communication for people with disabilities during Hurricane Harvey",
     "fact": "Barriers included lack of Text-to-911, system overloads, dependence on power, captioning failures, limited interpreter availability, and restricted access to information.",
     "fact_source": "Hurricane Harvey After Action Report on Individuals with Disabilities"},
    {"question": "What are the four Readiness Levels defined in the Brazos County Emergency Management Plan",
     "fact": "Level 4: Normal Conditions; Level 3: Increased Readiness; Level 2: High Readiness; Level 1: Maximum Readiness (e.g., tropical weather threat, tornado warning, flash flood).",
     "fact_source": "Brazos County Emergency Management Plan"},
    {"question": "What issues were identified regarding contractor safety during the storm Mara?",
     "fact": "Contractors exhibited unsafe behaviors. Recommended actions include pre-approval of contractor safety plans and formal processes to document, report, and correct unsafe behavior. A related finding noted a defective utility pole previously tagged as unsafe contributed to the Smokehouse Creek Fire.",
     "fact_source": "Winter Storm Mara: After Action Report"},
    {"question": "What is CRISP program in Austin?",
     "fact": "CRISP stands for the Community Resiliency Improvement Status Portal, developed by the Austin and Travis County Emergency Management Offices to provide transparency into disaster response and recovery efforts.",
     "fact_source": "Austin CRISP"},
    {"question": "How does the CRISP dashboard track the status of After-Action Report recommendations",
     "fact": "Statuses include Completed (verified finished), In Progress (with percentage completion), Not Started, On Hold or Pending (awaiting assignment or resources), and Closed (cannot be completed or merged into another item).",
     "fact_source": "Austin CRISP"},
    {"question": "What location was used for a full-scale county-wide POD exercise in July 2017 to simulate recovery from a Category 4 hurricane",
     "fact": "The NRG Center was used for the full-scale POD exercise.",
     "fact_source": "Harris County Tests Points of Distribution Plan"},
    {"question": "How did Hurricane Harvey flooding impact POD locations in Houston?",
     "fact": "Flooding made many POD sites and routes inaccessible, reduced effectiveness of pre-selected locations, and required post-incident site authorization. Studies indicate flood-aware POD planning could reduce travel distance by 46.5%. Infrastructure failures further constrained viable POD locations.",
     "fact_source": "https://pubmed.ncbi.nlm.nih.gov/32441037/"},
    {"question": "What specific equipment is required for a Type I POD?",
     "fact": "Required equipment includes 3 forklifts, 3 pallet jacks, 2 power light sets, 6 portable toilets, 2 tents, 4 dumpsters, 30 traffic cones, and 4 two-way radios.",
     "fact_source": "Harris County Points of Distribution (POD) Plan"},
    {"question": "What are the three core components used to assess flood risk in the 2024 State Flood Plan",
     "fact": "Flood risk is assessed using Flood Hazard (magnitude, location, frequency), Potential Exposure (people and property in hazard areas), and Vulnerability (degree of impact on communities and facilities).",
     "fact_source": "2024 State Flood Plan | Texas Water Development Board"},
    {"question": "How many Texans currently live in a 100-year floodplain?",
     "fact": "Approximately 2.4 million Texans live or work within the 1 percent (100-year) annual chance floodplain.",
     "fact_source": "2024 State Flood Plan | Texas Water Development Board"},
    {"question": "What is the Galveston Bay Surge Protection project's total cost?",
     "fact": "The Coastal Texas Project is authorized at $34.38B. The Galveston Bay Storm Surge Barrier System accounts for $31.20B, with a 65% federal share and 35% non-federal share.",
     "fact_source": "2024 State Flood Plan | Texas Water Development Board"},
    {"question": "How do playa improvements help reduce state flood risk?",
     "fact": "Playa improvements mitigate flat-terrain ponding through region-specific, nature-based and structural flood mitigation solutions.",
     "fact_source": "Technical Guidelines for Regional Flood Planning | TWDB"},
    {"question": "Tell me more about the strategic buyout program in Onion Creek.",
     "fact": "The voluntary buyout program addresses repeated flooding by purchasing homes along Upper Onion Creek, removing structures, restoring land through regrading and revegetation, and preserving it as permanent open space with potential community benefits.",
     "fact_source": "Nature-based solution Hill Country documentation"},
    {"question": "How does the flat terrain of the Houston area influence the choice of hydraulic modeling software used for flood planning",
     "fact": "Flat coastal terrain causes widespread, multi-directional flooding requiring 2D hydraulic analysis. Houston-area planning commonly uses HEC-RAS 2D or XPSWMM, with 1D/2D coupling or alternatives needed to represent underground drainage systems.",
     "fact_source": "Technical Guidelines for Regional Flood Planning | TWDB"},
]

In [ ]:
from tqdm import tqdm
import mlflow

mlflow.autolog(disable=True)

# Models to evaluate (subset of dbx_llms)
eval_llms = [
    "databricks-claude-opus-4-1",
    "databricks-claude-sonnet-4",
    "databricks-gpt-oss-20b",
    "databricks-llama-4-maverick",
    "databricks-gemma-3-12b",
]

for llm_model in tqdm(eval_llms, desc="LLMs"):
    for item in tqdm(qa_dict, desc=f"QA | {llm_model}", leave=False):
        q = item["question"]
        try:
            _, ans = rag_pipe_main(spark, q, llm_model=llm_model, temperature=0.2, use_dbx_model=True)
        except Exception:
            _, ans = rag_pipe_main(spark, q, llm_model=llm_model, use_dbx_model=True)
        item[f"answer_{llm_model}"] = ans

In [ ]:
import json
from pyspark.sql import Row

def _to_str(v):
    if v is None:
        return None
    if isinstance(v, (str, int, float, bool)):
        return str(v)
    return json.dumps(v, ensure_ascii=False)

def normalize_qa_dict(rows):
    out = []
    for r in rows:
        out.append({k.replace("-", "_"): _to_str(v) for k, v in r.items()})
    return out

qa_rows = normalize_qa_dict(qa_dict)
eval_df = spark.createDataFrame([Row(**x) for x in qa_rows])

EVAL_TABLE = "tdis_dev_data_catalog.tdir.qa_dict_eval"
eval_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(EVAL_TABLE)
display(eval_df)